In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import os

In [2]:
df = pd.read_csv("synthetic_core_pc_dataset.csv")
df.head()

,Curve_ID,Core_Porosity,Core_Permeability_mD,Reservoir_Porosity,Reservoir_Permeability_mD,Sw_1,Sw_2,Sw_3,Sw_4,Sw_5,...,Pc_1,Pc_2,Pc_3,Pc_4,Pc_5,Pc_6,Pc_7,Pc_8,Pc_9,Pc_10
0,1,0.1624,2940.067,0.2830,248.104,0.1665,0.1965,0.2955,0.2967,0.3199,...,108.955,68.559,30.372,29.461,27.229,14.858,12.974,12.340,11.592,11.039
1,2,0.1239,26.116,0.2481,1.534,0.2020,0.2281,0.2476,0.3937,0.5021,...,351.073,235.655,179.369,74.289,58.764,56.964,47.076,41.733,37.506,37.112
2,3,0.1655,1.885,0.3072,26.731,0.1544,0.2096,0.2627,0.3090,0.7155,...,65.961,44.025,35.827,32.422,20.175,20.144,19.983,19.209,19.103,18.911
3,4,0.2752,16.573,0.1299,712.819,0.1703,0.1751,0.2363,0.4015,0.4920,...,167.959,165.084,99.095,61.946,55.096,54.630,53.188,52.905,48.451,46.990
4,5,0.2764,34.270,0.3019,3840.300,0.1556,0.2459,0.3277,0.3323,0.4201,...,248.506,75.486,52.169,48.792,37.797,32.590,30.326,28.627,22.097,21.165


In [3]:
NUM_POINTS = 10
COMMON_POINTS = 50
common_sw = np.linspace(0.15,0.95,COMMON_POINTS)
plot_folder = "Interpolation_Check"
os.makedirs(plot_folder,exist_ok=True)

In [4]:
long_dataset = []

for index, row in df.iterrows():

    
    phi_core = max(row["Core_Porosity"], 1e-6)
    k_core = max(row["Core_Permeability_mD"], 1e-6)

    phi_res = max(row["Reservoir_Porosity"], 1e-6)
    k_res = max(row["Reservoir_Permeability_mD"], 1e-6)

    scaling = np.sqrt((k_core / phi_core) / (k_res / phi_res))

    
    Sw = []
    Pc = []

    for i in range(1, NUM_POINTS + 1):
        Sw.append(row[f"Sw_{i}"])
        Pc.append(row[f"Pc_{i}"])

    
    Sw = np.array(Sw, dtype=float)
    Pc = np.array(Pc, dtype=float)

    
    sort_idx = np.argsort(Sw)

    Sw = Sw[sort_idx]
    Pc = Pc[sort_idx]

    
    Sw_unique, unique_idx = np.unique(Sw,return_index=True )

    Pc_unique = Pc[unique_idx]

    
    interpolation = interp1d(
        Sw_unique,
        Pc_unique,
        kind="linear",
        bounds_error=False,
        fill_value=(
            Pc_unique[0],
            Pc_unique[-1]
        )
    )

    Pc_interp = interpolation(common_sw)

    
    Pc_interp = np.nan_to_num(
        Pc_interp,
        nan=Pc_unique[-1],
        posinf=Pc_unique[-1],
        neginf=Pc_unique[0]
    )

    
    Pc_upscaled = Pc_interp * scaling

    
    if (
        np.any(np.isnan(Pc_interp))
        or np.any(np.isinf(Pc_interp))
    ):
        print(f"Skipping Curve {int(row['Curve_ID'])}")
        continue

    
    for j in range(COMMON_POINTS):

        long_dataset.append({

            "Curve_ID": row["Curve_ID"],

            "Sw": common_sw[j],

            "Core_Pc": Pc_interp[j],

            "Upscaled_Pc": Pc_upscaled[j],

            "Core_Porosity": phi_core,

            "Core_Permeability_mD": k_core,

            "Reservoir_Porosity": phi_res,

            "Reservoir_Permeability_mD": k_res

        })

   
    if index < 20:

        plt.figure(figsize=(6,5))

        plt.scatter(
            Pc,
            Sw,
            label="Original Core"
        )

        plt.plot(
            Pc_interp,
            common_sw,
            label="Interpolated Core"
        )

        plt.plot(
            Pc_upscaled,
            common_sw,
            label="Upscaled"
        )

        plt.xlabel("Capillary Pressure (psi)")
        plt.ylabel("Water Saturation")
        plt.title(f"Curve {int(row['Curve_ID'])}")

        plt.grid(True)
        plt.legend()
        plt.tight_layout()

        plt.savefig(
            f"{plot_folder}/Curve_{int(row['Curve_ID'])}.png",
            dpi=300
        )

        plt.close()

In [5]:
long_df = pd.DataFrame(long_dataset)



In [6]:

long_df.to_csv("core_upscaled_dataset_long.csv", index=False)

